# AWS SageMaker Model Registry. Versions, Promotion and Rollback

Registering two trained model files as versions of one model, promoting a version from staging to production, serving whichever version is in production, and rolling back when a new version turns out worse than its numbers said.

---

## **Agenda**

1. [The project and the registry](#scrollTo=c003)
2. [Setup](#scrollTo=c004)
3. [Register version 1](#scrollTo=soMaj9xK0NjN)
4. [Promote version 1](#scrollTo=c006)
5. [Serve the production version](#scrollTo=Kmclo_vY0hrg)
6. [Release version 2](#scrollTo=c008)
7. [Roll back](#scrollTo=v1M9HhHt2CL7)
8. [Tear down](#scrollTo=c010)
9. [Class summary](#scrollTo=c011)
10. [References and further reading](#scrollTo=c012)

## **Preface**

A team that trains models soon has many model files. One of them is answering customers right now. Another passed its checks last week and is waiting. A third was tried and dropped. The MLflow class ended on this gap. Its tracking store held several logged models, and nothing recorded which one was the approved one.

A **model registry** is the place that records it. Each model gets one entry, every trained file becomes a numbered version of it, and each version carries a status. The status says whether the version may be deployed and how far it has got in the release.

This class uses the registry built into SageMaker AI. It starts from an empty account and a new project, hotel booking cancellations, with two model files that were trained earlier. You register both files as versions of one model. You promote the first from staging to production and put it behind an endpoint. Then you release the second, which scores better on paper. A check on real bookings shows it would hurt the hotel, so you roll production back to the first version, in the registry and on the endpoint.

**[Please Note:]** Everything happens in the browser. **Studio** is where the registry has its forms, and its **Deploy** button puts a version on the endpoint and switches it. The **SageMaker AI console** shows which version the endpoint is serving and deletes what Studio leaves behind, and the **S3 console** holds the model files. **CloudShell** is used for two small jobs. It looks up the address of the XGBoost container in section 3, and it runs the check on 1,000 bookings in sections 5 and 7. Each code block below is pasted into CloudShell as it is. Check that the region at the top right of the console says **United States (N. Virginia)** first. When CloudShell asks before pasting more than one line, click **Paste** in the **Safe Paste for multiline text** box, and press Enter if the last line has not run. Outputs come from a real run, so ids, times and dates in yours will differ. The endpoint bills by the hour from section 5 on, so do section 8 before you leave.

**By the end you will be able to do all of this.**

1. Say what a model group, a model version, an approval status and a lifecycle stage are, and which one controls deployment.
2. Register a model file that was trained somewhere else as a version, with its validation score attached.
3. Promote a version from staging to production, and know that only an approved version can be deployed.
4. Deploy the version that is in production with Studio's Deploy button, and ask the endpoint which version it is serving.
5. Release a new version to the same endpoint, and say why this kind of endpoint stops answering while it switches.
6. Roll back, in the registry and on the endpoint, when a release check fails.
7. Tear everything down in the right order.

**Prerequisites.** Real-time endpoints and the `model.tar.gz` file a training job leaves behind, from the SageMaker classes. AUC, and turning a probability into a yes or no with a cut-off.

## **1. The project and the registry**

### 1.1 The project

A hotel loses money on every room that stays empty because a guest cancelled late. If it knows which bookings are likely to be cancelled, it can take a few extra bookings for those nights, or call those guests to check. There is a cost on the other side too. A guest the hotel wrongly expects to cancel may arrive and find their room given to someone else.

The data is real. It holds 119,390 bookings at two hotels in Portugal, a city hotel and a resort, for arrivals from July 2015 to August 2017 (Antonio, de Almeida and Nunes, 2019). Each row is one booking as it looked when it was made, plus how it ended. The label is `is_canceled`. It is 1 if the booking was cancelled and 0 if the guest came, and 37% of the bookings were cancelled.

Here are three bookings for the end of August 2017, with 12 of the 32 columns.

```
                              booking 1     booking 2     booking 3
hotel                      Resort Hotel  Resort Hotel  Resort Hotel
lead_time                             0           172            17
arrival_date_month               August        August        August
arrival_date_day_of_month            26            26            26
stays_in_week_nights                  1             1             1
adults                                2             3             2
country                             PRT           PRT           USA
market_segment                   Direct        Direct     Online TA
deposit_type                 No Deposit    No Deposit    No Deposit
adr                                 235           207           240
reservation_status             Canceled      Canceled      Canceled
is_canceled                           1             1             1
```

`lead_time` is the number of days between booking and arrival. `adr` is the average price per night in euros. `market_segment` says where the booking came from, such as `Online TA`, an online travel agent. Two columns, `reservation_status` and `reservation_status_date`, record how the booking ended. They give the answer away, so the models never see them. That leaves 29 input columns.

The models were trained with SageMaker's built-in XGBoost container, which reads CSV with numbers only and no header. So every text value was swapped for a number before training. These are the same three bookings as the model receives them.

```
1,0,2017,1,34,26,0,1,2,0,0,0,129,3,1,0,0,0,6,6,0,0,0,0,0,2,235.0,0,0
1,172,2017,1,34,26,0,1,3,0,0,0,129,3,1,0,0,0,0,0,0,0,250,0,0,2,207.0,0,0
1,17,2017,1,34,26,1,1,2,0,0,0,161,6,3,0,0,0,3,3,0,0,240,0,0,2,240.0,0,2
```

The columns are in the same order as the original file, without the label and the two outcome columns. Each text value has a fixed number.

| Column (position) | Value | Number |
|---|---|---|
| `hotel` (1) | Resort Hotel | 1 |
| `arrival_date_month` (4) | August | 1 |
| `country` (13) | PRT, USA | 129, 161 |
| `market_segment` (14) | Direct, Online TA | 3, 6 |

The numbers follow alphabetical order, which is why August is 1. They are codes, not amounts. The full list is in `data/encoding.json` in the model repository below.

### 1.2 What the registry holds

The registry has four ideas. The names in brackets are what the API calls them.

- A **model group** (model package group) is one entry per problem. Here it is `hotel-cancellations`.
- A **model version** (model package) is one trained file registered in the group. It records where the file is in S3, which container serves it, and any facts you attach, such as the validation AUC. Versions are numbered 1, 2, 3 in order, and a number is never used twice. The file itself stays in S3. The registry only points at it.
- The **approval status** is Pending, Approved or Rejected. Studio shows it as **Deployment status**. SageMaker refuses to make a deployable model from a version that is not Approved, so this is the one setting that controls deployment.
- The **lifecycle stage** says where the version is in the team's release, such as PreProduction or Production. Studio's list has no "Staging". PreProduction is SageMaker's word for it. The stage is a record for people. SageMaker does not act on it.

| Piece | In this class | What it decides |
|---|---|---|
| Model group | `hotel-cancellations` | Which problem the versions belong to |
| Model version | Versions 1 and 2 | Which file and container would serve |
| Approval status | Pending, Approved, Rejected | Whether the version can be deployed at all |
| Lifecycle stage | PreProduction, Production, Archived | Where the version is in the release |

The registry does not serve anything. An endpoint serves one version, and keeping the endpoint and the registry in step is your job. In big teams a pipeline does it. Section 5 shows how to ask an endpoint which version it is serving.

### 1.3 The two model files

Both files were made by SageMaker training jobs, the way the training class did it. They learned from bookings that arrived before June 2017 and were scored on bookings for June and July 2017. They are in a public repository, [pulkit-scaler/mlops-model-registry](https://github.com/pulkit-scaler/mlops-model-registry), with a README that says how each was trained.

| Setting | Version 1 | Version 2 |
|---|---|---|
| Trees | 100, each 4 levels deep | 300, each 6 levels deep |
| Weight on cancelled bookings | Normal | 3 times |
| Validation AUC | 0.8817 | 0.9028 |

Version 2 is bigger, and each cancelled booking counted three times while it trained. Teams do that when catching cancellations matters most to them. On validation AUC, version 2 is clearly better.

## **2. Setup**

### 2.1 A Studio domain with quick setup

The registry lives in SageMaker AI, and its forms are in Studio, so the class needs a **domain**. A domain is what Studio runs in, with an **execution role** that Studio acts as. Quick setup makes both in under a minute. It also makes the bucket `sagemaker-us-west-2-` followed by your account number, which is where the model files go.

**[Please Note:]** Browser. Type `SageMaker AI` in the console search bar and open it. In the left menu, under **Environment configuration**, click **Domains**, then **Create domain**. Keep **Set up for single user (Quick setup)** and click **Set up**. If the account already has a domain from an earlier class, skip this and use that one.

![c15_quick_setup.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/249/976/original/c15_quick_setup.png?1793174627)

Quick setup puts the domain in the account's **default VPC**, the network every account starts with. A cleanup script can delete it. Without it, quick setup stops with this error.

![c15_no_vpc_error.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/249/977/original/c15_no_vpc_error.png?1793174630)

**[Please Note:]** Only if you see that error. Search for `VPC`, open **Your VPCs**, choose **Actions → Create default VPC** and confirm. Then go back to SageMaker AI and click **Set up** again. The failed attempt has already made a set of roles, and the second attempt makes another, so section 8 has an extra set to remove.

With the default VPC in place, the domain was ready 18 seconds after the click. Open it from the **Domains** list. Its **Status** says **Ready**, and the **User profiles** tab lists the profile quick setup made, named `default-` followed by the date and time.

![c15_domain_ready.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/156/original/c15_domain_ready.png?1793207537)

### 2.2 Copy the model files into S3

A version points at a file in S3, so the two files have to be in S3 before they can be registered. They come from the model repository on GitHub, and the S3 console uploads them into the bucket quick setup made.

**[Please Note:]** Browser.

1. Open [pulkit-scaler/mlops-model-registry](https://github.com/pulkit-scaler/mlops-model-registry), click **Code**, then **Download ZIP**, and unzip it on your laptop. Inside is a `models` folder with `v1/model.tar.gz` and `v2/model.tar.gz`.
2. Type `S3` in the console search bar and open it. Under **General purpose buckets**, click the bucket whose name starts with `sagemaker-us-west-2-`.
3. Click **Create folder**, name it `hotel` and click **Create folder**. Click `hotel/` to open it.
4. Click **Upload**, then **Add folder**, and choose the `models` folder from the ZIP. Click **Upload**.

![c15_s3_uploaded.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/169/original/c15_s3_uploaded.png?1793207574)

Both files are in, under `hotel/models/v1/` and `hotel/models/v2/`. The folders keep the two files apart, since both are called `model.tar.gz`. Version 2's file is ten times bigger, 397.5 KB against 39.2 KB, because it has three times as many trees and each one is deeper. The repository's README says what is inside each archive, one file called `xgboost-model`, which is the trained model and nothing else.

Studio's form in 3.2 needs each file's S3 address. The S3 console gives it to you.

**[Please Note:]** S3 console. Open `models/`, then `v1/`, tick `model.tar.gz` and click **Copy S3 URI**. Keep the S3 tab open, because 6.1 needs the same for `v2/`.

![c15_s3_v1.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/170/original/c15_s3_v1.png?1793207577)

## **3. Register version 1**

### 3.1 Open Studio

**[Please Note:]** Browser. Go back to the SageMaker AI tab. In the left menu, under **Applications and IDEs**, click **SageMaker Studio**. Pick the `default-` user profile and click **Open Studio**. Studio opens in a new tab. Close any tour it offers. Home shows a banner, **Discover the new "Models" section**. Close it with its **X**, because it covers the **Models** menu.

![c15_studio_home.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/249/979/original/c15_studio_home.png?1793174635)

### 3.2 The group and its first version

Studio makes the group and its first version in one form. **Bring your own model** is the choice for a file trained somewhere else. It needs two things. The **artifact source** is the file in S3. The **container** is the image that will serve the file.

These files were trained with AWS's XGBoost container, version 1.7-1. They must be served by the same container, and the form asks for its address. You have seen this address before. In the training class, `image_uris.retrieve("xgboost", REGION, "1.7-1")` gave it to you. That function reads the address from a list that comes inside the SageMaker Python SDK. CloudShell does not have the SDK, so the command below downloads the same list from GitHub and picks out the address.

**[Please Note:]** Browser, the SageMaker AI tab. Click the terminal icon in the top bar to open CloudShell. Close the welcome box if it shows. Wait for the `~ $` prompt, then paste the command.

In [ ]:
curl -s https://raw.githubusercontent.com/aws/sagemaker-python-sdk/master/sagemaker-core/src/sagemaker/core/image_uri_config/xgboost.json \
  | jq -r '.inference.versions["1.7-1"] | "\(.registries["us-west-2"]).dkr.ecr.us-west-2.amazonaws.com/\(.repository):1.7-1"'

Output from the run.

```
683313688378.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.7-1
```

This is the container's address. Copy it. You paste it into the form in step 4 below. Each part of the address means something.

- `683313688378` is the AWS account that stores the container. It is AWS's own account, not yours.
- `us-west-2` is the region. Other regions keep the container under a different account number, so this address works only in us-west-2.
- `sagemaker-xgboost` is the container's name.
- `1.7-1` is its version.

The validation AUC goes in as **custom metadata**. The registry was not there when these files were trained, so it knows nothing about how good they are unless you tell it. Custom metadata is the place for that.

**[Please Note:]** Studio.

1. In the left menu, click **Models** to open it, then **Custom models**. The **Logged** tab is empty. Click **Register model group**.
2. Under **Model**, keep **Create model group**. Replace the name it suggests with `hotel-cancellations`. As the description, type `Predicts whether a hotel booking will be cancelled`.
3. Under **Artifact**, click **Bring your own model**.
4. **Artifact source** is the S3 address you copied in 2.2. It ends in `hotel/models/v1/model.tar.gz`. In **Container (ECR URI)**, paste the address CloudShell printed.
5. Leave **Model approval status** at **Pending**.
6. Under **Custom metadata**, type `validation_auc` as the name and `0.8817` as the value.
7. Click **Register**.

![c15_register_v1.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/166/original/c15_register_v1.png?1793207566)

One click made both the group and version 1, and Studio opened the new version's page.

**[Please Note:]** If that page says **Something went wrong**, reload the browser tab.

### 3.3 What a version holds

**[Please Note:]** Studio. On the `hotel-cancellations` card, click **View all versions**.

![c15_versions_v1.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/181/original/c15_versions_v1.png?1793207607)

Version 1 is there, its deployment status is **Pending manual approval**, and its stage is empty. Tick its box and look at the buttons above the table. **Deploy** is grey. Studio deploys only an approved version, and approving is section 4.

![c15d_v1_pending_deploy.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/132/original/c15d_v1_pending_deploy.png?1793691911)

Untick the box, then click **Version 1** to open it, click the **Details** tab and scroll down to **Custom metadata**. The AUC you typed is there.

![c15_v1_metadata.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/178/original/c15_v1_metadata.png?1793207598)

The container address you pasted is on this page too.

**[Please Note:]** Studio, still on Version 1. Click the **Lineage** tab, then click the **Image** box. The address shows on the right, under **Source URI**. It should be the same address CloudShell printed.

![c15d_v1_lineage_image.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/119/original/c15d_v1_lineage_image.png?1793691869)

That is what a version is, a record that points at a file in S3 and a container, plus its status and the facts you attached. Registering copied nothing. The file itself stays where 2.2 put it. If someone deleted it, the version would still be listed but could no longer be served.

## **4. Promote version 1**

Promotion here means three changes to the version's record. It moves to staging while it is checked, it gets approved, and then it moves to production.

### 4.1 Staging

A team usually checks a version in staging before approving it. That can be a test endpoint, a review of the numbers, or both. Here the check is the validation AUC in the metadata. Version 1 is the first version, so there is nothing to compare it with, and its AUC of 0.8817 is the bar for later versions.

**[Please Note:]** Studio, the versions table of `hotel-cancellations`.

1. Tick **Version 1**, then click the **⋮** button at the right end of the table's toolbar, next to **Deploy**. Choose **Update model lifecycle**.
2. Set **Stage** to **PreProduction** and **Stage status** to **InProgress**. As the **Stage description**, type `Checking v1 before release`. Click **Save**.

![c15_lifecycle_staging.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/163/original/c15_lifecycle_staging.png?1793207558)

### 4.2 Approve, then production

**[Please Note:]** Studio, the same table.

1. Tick **Version 1**, click **⋮** and choose **Update deployment status**. Set **Status** to **Approved**, type `First version, validation AUC 0.8817` as the **Comment** and click **Save**.
2. Tick **Version 1** again, click **⋮** and choose **Update model lifecycle**. Set **Stage** to **Production** and **Stage status** to **Approved**, type `First release` and click **Save**.

![c15_approve_v1.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/155/original/c15_approve_v1.png?1793207534)

![c15_versions_v1_prod.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/180/original/c15_versions_v1_prod.png?1793207603)

Version 1 is now Approved and in Production. Open it and click the **Activity** tab. Every change is there, with its comment, who made it and when. The two oldest rows come from registering it. That history is most of what a registry is for. When something goes wrong in production, the first question is always what changed and who changed it.

![c15_v1_activity.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/177/original/c15_v1_activity.png?1793207595)

## **5. Serve the production version**

### 5.1 Deploy from the registry

Serving a version takes the same three objects as in the endpoint classes. They are a model, an endpoint configuration and an endpoint. Studio's **Deploy** button makes all three from one form. It makes the model from the version itself, so the model gets the version's file and container from the registry. The button works only on an Approved version, which is why version 1 was approved first.

The endpoint it makes is a newer kind than the one in the endpoint classes. The endpoint holds the machine, and the model runs on it as an **inference component**. A component is one model plus the share of the machine's CPU and memory it may use. Studio names the component after the group, then two numbers and the date, for example `hotel-cancellations-1793686347230-20261103-0612350`. That name matters when you call the endpoint in 5.2.

**[Please Note:]** Studio, the versions table of `hotel-cancellations`.

1. Tick **Version 1** and click **Deploy** in the toolbar. The form takes about 15 seconds to fill in.
2. Open the **Endpoint name** list and choose **Enter endpoint name**. In **Custom endpoint name**, type `hotel-cancel`.
3. Set **Instance type** to **ml.m5.large**. The form suggests a bigger machine, and not always the same one, so always check this field.
4. Set **Maximum instance count** to `1`. It starts at 20, which would let the endpoint grow to 20 machines when it is busy. Leave **Initial instance count** at 1.
5. Leave the **Models** section as it is and click **Deploy**.

![c15d_deploy_form_v1.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/125/original/c15d_deploy_form_v1.png?1793691894)

The model in the **Models** section, `hotel-cancellations-` followed by a number, was made the moment the form opened. Studio makes a new one every time this form opens, even if you click **Cancel**. Section 8 deletes them all.

Studio then opens the endpoint's page. You can come back to it from **Deployments → Endpoints** in Studio's left menu. The endpoint and its component show **Creating**. In this run the component was **In service** about five minutes after the click. Wait for that before 5.2.

![c15d_endpoint_page_v1.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/127/original/c15d_endpoint_page_v1.png?1793691899)

### 5.2 A check on real bookings

A good validation score does not prove a model will help. The hotel can check that on bookings whose ending it already knows. The repository has 1,000 bookings for the last week of August 2017, which is after both models' training and validation data. Each one has its real ending in `live_labels.csv`.

The check sends all 1,000 to the endpoint in one call, the rows exactly as section 1.1 showed them. A booking counts as **flagged** if its chance of cancelling is above 0.5. If the hotel gave away the room of every flagged booking, every flagged guest who still came would arrive to no room. So the check counts those too.

Calling this endpoint needs one thing more than in the endpoint classes. An endpoint with components has to be told which component should answer. Without the component's name SageMaker refuses the call, saying that the "Inference Component Name header is required". The name changes with every deploy, so the check does not type it in. It first asks SageMaker which component is in service on `hotel-cancel`, prints that name, and sends the bookings to it.

The block downloads the two files, saves the check as `check_live.sh`, and runs it once. The console has no page that can send a file of bookings to an endpoint and count the answers, which is why this stays in CloudShell.

**[Please Note:]** Browser, CloudShell in the SageMaker AI tab, which you opened in 3.2. If it is closed, click the terminal icon in the top bar again. Paste the block. Next time, `bash ~/hotel/check_live.sh` is enough.



In [ ]:
mkdir -p ~/hotel && cd ~/hotel
REPO=https://raw.githubusercontent.com/pulkit-scaler/mlops-model-registry/main
curl -sSf -o live.csv ${REPO}/data/live.csv
curl -sSf -o live_labels.csv ${REPO}/data/live_labels.csv
cat > check_live.sh <<'END'
# Score the 1,000 known bookings with whatever version is live, and compare with what really happened.
cd ~/hotel
IC=$(aws sagemaker list-inference-components --endpoint-name-equals hotel-cancel --status-equals InService \
  --query 'InferenceComponents[0].InferenceComponentName' --output text)
echo "component ${IC}"
aws sagemaker-runtime invoke-endpoint --endpoint-name hotel-cancel --inference-component-name ${IC} \
  --content-type text/csv --body fileb://live.csv answers.txt > /dev/null
echo "first three answers"; head -3 answers.txt
paste -d, answers.txt live_labels.csv | awk -F, '
  $2 == 1  { cancelled++ }
  $1 > 0.5 { flagged++; if ($2 == 0) came++ }
  END { print "bookings", NR, " really cancelled", cancelled
        print "flagged", flagged, " flagged but the guest came", came }'
END
bash check_live.sh

In [ ]:

import boto3
import configparser
import requests
import numpy as np
import pandas as pd
from io import StringIO
from pathlib import Path

endpoint_name = "hotel-cancel"
repo = "https://raw.githubusercontent.com/pulkit-scaler/mlops-model-registry/main"

session = boto3.Session()

if session.get_credentials() is None:
    config_path = Path("aws_config.cfg")

    if not config_path.exists():
        raise FileNotFoundError(f"AWS credentials not found. Missing {config_path}")

    config = configparser.ConfigParser()
    config.read(config_path)

    aws_config = config["AWS"]

    session = boto3.Session(
        aws_access_key_id=aws_config["aws_access_key_id"],
        aws_secret_access_key=aws_config["aws_secret_access_key"],
        aws_session_token=aws_config.get("aws_session_token"),
        region_name=aws_config.get("region_name", session.region_name)
    )

sagemaker = session.client("sagemaker")
runtime = session.client("sagemaker-runtime")

live_response = requests.get(f"{repo}/data/live.csv", timeout=30)
labels_response = requests.get(f"{repo}/data/live_labels.csv", timeout=30)

live_response.raise_for_status()
labels_response.raise_for_status()

live_csv = live_response.content
labels = pd.read_csv(
    StringIO(labels_response.text), header=None
).iloc[:, 0].to_numpy(dtype=int)

components = sagemaker.list_inference_components(
    EndpointNameEquals=endpoint_name,
    StatusEquals="InService"
)["InferenceComponents"]

if len(components) != 1:
    raise ValueError(
        f"Expected one active inference component, found {len(components)}: "
        f"{[c['InferenceComponentName'] for c in components]}"
    )

ic_name = components[0]["InferenceComponentName"]
print(f"Component: {ic_name}")

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    ContentType="text/csv",
    Body=live_csv
)

prediction_text = response["Body"].read().decode("utf-8")
predictions = np.fromstring(
    prediction_text.replace(",", " "), sep=" "
)

if len(predictions) != len(labels):
    raise ValueError(
        f"Prediction count ({len(predictions)}) "
        f"does not match label count ({len(labels)})"
    )

print("First three answers:", predictions[:3])

flagged = predictions > 0.5
cancelled = labels == 1

print("Bookings:", len(labels))
print("Really cancelled:", np.sum(cancelled))
print("Flagged:", np.sum(flagged))
print("Flagged but guest came:", np.sum(flagged & ~cancelled))


Output from the run.

```
component hotel-cancellations-1793686347230-20261103-0612350
first three answers
0.20676228404045105
0.6976045370101929
0.3628968596458435
bookings 1000  really cancelled 298
flagged 239  flagged but the guest came 78
```

The first line is the component the check used. The endpoint answers one chance of cancellation per booking. The first three are the three bookings from section 1.1, at 0.21, 0.70 and 0.36. All three were in fact cancelled, and at a cut-off of 0.5 version 1 flags only the second. Over all 1,000, 298 were really cancelled. Version 1 flagged 239, and 78 of those guests came anyway. That is the baseline. Every later version gets the same check.

### 5.3 Which version is live

The registry says version 1 is in production. Nothing forces the endpoint to agree. So ask the endpoint. Studio's endpoint page names the component and stops there. The SageMaker AI console follows the whole trail, from the endpoint to its component, the component's model, and the version that model was made from.

**[Please Note:]** Browser, the SageMaker AI tab. In the left menu, open **Deployments & inference** and click **Endpoints**. Click `hotel-cancel`, then the **Inference Components** tab. In the component's card, click the name under **Model**.

![c15d_console_ics_v1.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/120/original/c15d_console_ics_v1.png?1793691880)

![c15d_console_model_v1.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/122/original/c15d_console_model_v1.png?1793691887)

The component runs on an ml.m5.large, and its model is the `hotel-cancellations-` model from 5.1. That model's **Container 1** is **Model Package** `hotel-cancellations/1`, the same version the registry has in Production.

Do not look for the deployment in Studio's **Activity** tab. With this kind of endpoint the tab does not list it, so version 1's history still ends at its move to Production. That is one more reason to ask the endpoint.

## **6. Release version 2**

### 6.1 Register it

Version 2 goes into the same group. A second form fill is all it takes, and the registry numbers it 2.

**[Please Note:]** Studio. In the left menu, click **Models**, then **Custom models**, then **Register model group**.

1. Under **Model**, click **Find model group** and click the round button in the `hotel-cancellations` row. If the page says **Error loading module**, reload the browser tab and start again.
2. Under **Artifact**, click **Bring your own model**. **Artifact source** is version 2's S3 address. In the S3 tab, open `models/`, then `v2/`, tick `model.tar.gz` and click **Copy S3 URI**. It ends in `hotel/models/v2/model.tar.gz`. In **Container (ECR URI)**, paste the same XGBoost address as before.
3. Leave **Model approval status** at **Pending**. Under **Custom metadata**, type `validation_auc` and `0.9028`.
4. Click **Register**.

![c15_register_v2.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/167/original/c15_register_v2.png?1793207568)

### 6.2 Promote it, and archive version 1

Version 2 goes through the same three steps as version 1. Its AUC of 0.9028 beats version 1's 0.8817, so it passes the staging review. When it reaches Production, version 1 moves to **Archived**, so only one version is in Production.

**[Please Note:]** Studio, the versions table of `hotel-cancellations`. Use **⋮** at the right end of the toolbar each time, with only the version named ticked.

1. **Version 2**, **Update model lifecycle**. **PreProduction**, **InProgress**, `Checking v2 before release`, **Save**.
2. **Version 2**, **Update deployment status**. **Approved**, `Validation AUC 0.9028, better than v1`, **Save**.
3. **Version 2**, **Update model lifecycle**. **Production**, **Approved**, `Second release`, **Save**.
4. **Version 1**, **Update model lifecycle**. **Archived**, **Approved**, `Replaced by v2`, **Save**.

![c15_versions_v2_prod.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/182/original/c15_versions_v2_prod.png?1793207609)

### 6.3 Move the endpoint to version 2

The registry has moved. The endpoint has not. It serves version 1 until it is told otherwise.

The Deploy form can add a model to an endpoint that already exists. But an endpoint like this one holds only one registry version at a time. While version 1's component is on it, Studio refuses version 2, saying that "An endpoint with Inference Components enabled supports only one model package". So the switch takes two steps. First remove version 1's component, then deploy version 2 onto the same endpoint. The endpoint keeps its name and its machine, so anything that calls `hotel-cancel` never has to change.

Between the two steps nothing is serving. In this run the endpoint answered nothing for about seven minutes. Nearly three of those went on the delete, under two on the form, and about two and a half on starting version 2. For a hotel's booking site that is an outage. The single-model endpoints from the endpoint classes avoid it, because SageMaker starts the new version before it stops the old one. With the Deploy button, plan a switch for a quiet hour.

**[Please Note:]** Studio. In the left menu, open **Deployments** and click **Endpoints**, then `hotel-cancel`. On the **Models** tab, click the round button of the one component, click **Delete** above the table, then **Delete model**.

![c15d_delete_component.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/124/original/c15d_delete_component.png?1793691891)

The component shows **Deleting**. Wait until the table is empty. It took a little under three minutes.

**[Please Note:]** Studio, the versions table of `hotel-cancellations`.

1. Tick **Version 2** and click **Deploy**.
2. Open the **Endpoint name** list and choose **Select an existing endpoint**. In the box, click the round button of `hotel-cancel`, then **Select**.
3. The machine settings turn grey, because they belong to the endpoint. **Maximum instance count** shows 20, but the endpoint keeps the 1 you set in 5.1. Click **Deploy**.

![c15d_existing_endpoint.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/128/original/c15d_existing_endpoint.png?1793691901)

![c15d_deploy_form_v2.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/126/original/c15d_deploy_form_v2.png?1793691897)

Version 2's component was in service about two and a half minutes after the click. It has a new name, and so does its model, which the form made when it opened. Follow the trail from 5.3 again. It now ends at **Model Package** `hotel-cancellations/2`.

![c15d_console_model_v2.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/123/original/c15d_console_model_v2.png?1793691889)

### **Checkpoint 6.1**

Open version 2 in Studio and use only its **Activity** tab. Write down who approved it, when, and with what comment, and match each row to the step in 6.1 or 6.2 that made it. Then do the same for version 1, whose rows go back to section 3. Notice that no deploy left a row, not 5.1 and not 6.3, so the tab tells you what the registry decided, not what the endpoint is serving. When a release goes wrong, this tab is the first place a team looks.

## **7. Roll back**

### 7.1 The release check

Version 2 has been live for a moment. Run the same check as in 5.2.

**[Please Note:]** CloudShell.

In [ ]:
bash ~/hotel/check_live.sh

Output from the run.

```
component hotel-cancellations-1793688096967-20261103-0641440
first three answers
0.3879683315753937
0.855282723903656
0.5738815665245056
bookings 1000  really cancelled 298
flagged 513  flagged but the guest came 241
```

| Check on 1,000 bookings | Version 1 | Version 2 |
|---|---|---|
| Really cancelled | 298 | 298 |
| Flagged above 0.5 | 239 | 513 |
| Flagged, but the guest came | 78 | 241 |

The check found version 2's component on its own. Version 2 flags more than twice as many bookings. Many of the extra ones are guests who came. If the hotel gave their rooms away, 241 guests would arrive to no room, against 78 with version 1.

The higher AUC did not lie. AUC only measures whether cancelled bookings get higher scores than kept ones. It ignores where the 0.5 cut-off falls. Counting each cancelled booking three times in training pushed version 2's scores up, so far more of them now land above 0.5. The first three answers show it, 0.39, 0.86 and 0.57 against 0.21, 0.70 and 0.36. Version 2 ranks bookings better and still gives the hotel worse decisions at the cut-off the hotel uses.

Moving the cut-off for version 2 would be a fair fix, but it is a change to how the hotel uses the model, and it needs its own checks. Right now the safe move is to put back what worked.

### 7.2 Endpoint first

Guests are affected now, so the endpoint goes back first. It takes the same two steps as 6.3 with the versions swapped, and it has the same gap. The registry still holds version 1, still approved, so nothing has to be registered again. Studio only makes a new model and a new component from it.

**[Please Note:]** Studio. On `hotel-cancel`'s **Models** tab, delete version 2's component as in 6.3 and wait until the table is empty. Then, in the versions table, tick **Version 1**, click **Deploy**, choose **Select an existing endpoint**, pick `hotel-cancel` and click **Deploy**. Do 7.3 while version 1 starts.

In this run the endpoint answered nothing for about seven minutes again, from the delete until version 1 was back in service.

### 7.3 Then the registry

The registry has to tell the truth about what happened, or the next person will promote version 2 again.

**[Please Note:]** Studio, the versions table of `hotel-cancellations`.

1. **Version 2**, **Update deployment status**. **Rejected**, `Flags 513 of 1,000 bookings, 241 of those guests came`, **Save**.
2. **Version 2**, **Update model lifecycle**. **Archived**, **Rejected**, `Rolled back`, **Save**.
3. **Version 1**, **Update model lifecycle**. **Production**, **Approved**, `Rolled back from v2`, **Save**.

The table now shows the rollback. With version 2 ticked, **Deploy** is grey.

![c15d_v2_rejected.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/133/original/c15d_v2_rejected.png?1793691914)

Rejected is not only a label. Studio's **Deploy** button stays grey for a Rejected version, as it did for a Pending one, and SageMaker refuses to make a new model from it. Version 2 can only be deployed again if someone approves it again, and that would show up in its Activity tab. The gate only covers new models, though. The model Studio made for version 2 in 6.3 is still in the account until section 8 deletes it.

### 7.4 Check the rollback

Version 1 was in service again about two and a half minutes after the click. The trail from 5.3 ends at **Model Package** `hotel-cancellations/1` again.

![c15d_console_model_back.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/121/original/c15d_console_model_back.png?1793691883)

Run the check once more.

**[Please Note:]** CloudShell.

In [ ]:
bash ~/hotel/check_live.sh

Output from the run.

```
component hotel-cancellations-1793688763954-20261103-0652540
first three answers
0.20676228404045105
0.6976045370101929
0.3628968596458435
bookings 1000  really cancelled 298
flagged 239  flagged but the guest came 78
```



The component is new, but the answers and the counts, 239 and 78, are the same as before version 2, because it is the same file. The registry and the endpoint agree again. Version 1's Activity tab tells the registry's side of the story, from first release to rollback.

![c15d_v1_activity_final.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/131/original/c15d_v1_activity_final.png?1793691909)

## **8. Tear down**

The endpoint is the only thing in this class that bills by the hour, so it goes first. Deleting it in Studio removes its component too. The endpoint configuration and the models Studio made stay behind, and only the SageMaker AI console lists them. Models and configurations cost nothing, but leaving them makes the next person's console harder to read. Last come the registry and the files. A group cannot be deleted while it still holds versions, so the versions go first.

**[Please Note:]** Studio, **Deployments → Endpoints**, `hotel-cancel`. Click **Delete** at the top right, tick **I confirm that I want to delete the endpoint** and click **Delete endpoint**.

![c15d_td_endpoint.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/129/original/c15d_td_endpoint.png?1793691904)

The endpoint went at once, and its component about a minute later.

**[Please Note:]** SageMaker AI console, under **Deployments & inference**. Each list selects one row at a time. Every delete box warns that Studio made the item. Tick the box and click **Delete**.

1. **Endpoint configurations**. Delete the one named `hotel-cancel-` followed by a number. Studio made it in 5.1.
2. **Deployable models**. Delete every model whose name starts with `hotel-cancellations-`. There is one for each time the Deploy form opened, three in this run.

![c15d_td_models.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/251/130/original/c15d_td_models.png?1793691907)

**[Please Note:]** Studio, the versions table of `hotel-cancellations`.

1. Tick both versions, click **⋮** at the right of the toolbar and choose **Delete**. Tick **Yes, delete the model versions.** and click **Delete**.
2. Go to **Models → Custom models**. On the `hotel-cancellations` card, click **⋮** and choose **Delete**, then **Delete** again.

![c15_td_group_menu.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/172/original/c15_td_group_menu.png?1793207582)

**[Please Note:]** S3 console, the `sagemaker-us-west-2-` bucket. Tick `hotel/`, click **Delete**, type `permanently delete` and click **Delete objects**. Stay on the page until its **Close** button turns on, because the S3 console deletes a folder from the browser. In CloudShell, `rm -rf ~/hotel` removes the files the checks left there.

![c15_td_s3.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/173/original/c15_td_s3.png?1793207584)

Check it in the console. In SageMaker AI, open **Deployments & inference** in the left menu and click **Endpoints**. The list is empty.

![c15_endpoints_empty.png](https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/250/162/original/c15_endpoints_empty.png?1793207556)

What is left costs nothing.

| Left behind | Costs | To remove it |
|---|---|---|
| The domain and its user profile | Nothing while no app runs | Delete the profile, then the domain |
| The execution role and two Canvas roles, a set for each quick setup attempt | Nothing | Delete them in IAM after the domain is gone |
| The bucket, now empty | Nothing | Delete it in S3 |
| The default VPC | Nothing | Leave it, quick setup needs it |

The next class opens Canvas from the same domain, so keep it if you can.

## **9. Class summary**

A registry records which model file is which, and which one is allowed to serve. Two files that were trained earlier became versions 1 and 2 of one group, `hotel-cancellations`. Each version holds an address in S3, a container and the facts you attach, such as its validation AUC. Studio added the history, with every approval and stage move.

Only one setting is a real gate. SageMaker deploys only an approved version, and Studio's Deploy button stays grey for any other. The lifecycle stage, PreProduction then Production, is a record for people. And the endpoint never reads the registry. It served whatever model it was given, and the console's trail from the endpoint to its component, its model and the model package proved which version that was.

Version 2 won on validation AUC and still lost on the hotel's real question. At the 0.5 cut-off it flagged 513 of 1,000 bookings, and 241 of those guests came, against 78 for version 1. AUC measures ranking and says nothing about where scores fall against a cut-off. Rolling back meant deploying version 1 again from the registry, which still held it, approved. The Deploy button made each switch a few clicks, and each switch cost about seven minutes with no answers, because this kind of endpoint holds one registry version at a time.

The check that caught version 2 could have run before release, in staging, on a test endpoint. That is what staging is for, and it is the habit to take away.

## **10. References and further reading**

**Official documentation**

- [Model Registration Deployment with Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html). Model groups, versions, and how they fit together.
- [Update the Approval Status of a Model](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-approve.html). The three statuses and what approval allows.
- [Staging Construct for your Model Lifecycle](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-staging-construct.html). The lifecycle stage and stage status, and how teams use them.
- [Deploy a Model from the Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-deploy.html). Making a model from a version, with the SDK, Boto3 or the console.
- [Deploy models for real-time inference](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-deploy-models.html). Endpoints with inference components, the kind Studio's Deploy button makes, and how they differ from single-model endpoints.
- [XGBoost container addresses](https://github.com/aws/sagemaker-python-sdk/blob/master/sagemaker-core/src/sagemaker/core/image_uri_config/xgboost.json). The list inside the SageMaker Python SDK that section 3 reads, with the address for every region and version.

**The data**

- Nuno Antonio, Ana de Almeida, Luis Nunes, [Hotel booking demand datasets](https://doi.org/10.1016/j.dib.2018.11.126), *Data in Brief* 22 (2019) 41-49. Licensed CC BY 4.0.
- [pulkit-scaler/mlops-model-registry](https://github.com/pulkit-scaler/mlops-model-registry). The two model files, the 1,000 bookings and the encoding used in this class.

**Practitioner reading**

- [MLflow Model Registry](https://mlflow.org/docs/latest/ml/model-registry/). The open source registry most teams meet first. It has versions and aliases such as `champion` where SageMaker has approval and stages.

**Pricing**

- [Amazon SageMaker AI pricing](https://aws.amazon.com/sagemaker/ai/pricing/), under real-time inference. The registry itself costs nothing.